# Pre-trend diagnostic — PT_violation_g05 (linearity_degree=2)

**Workstream PT · conditional-parallel-trends diagnostic (Reviewer 3.2.3)**

treated group on a differential path, slope 0.05 per period: detection power against violation magnitude

The estimation model sets `Z = D_it`, which is zero on every pre-treatment row,
so those rows carry **no information about tau** and no post-processing of that
fit can produce a placebo coefficient. This notebook therefore fits the
*unconstrained* specification (`Z = 1[G_i != inf]`, on in every period) as a
**separate diagnostic model**, and reports

$$\\Delta(k) = E\\big[\\tau(X, k) - \\tau(X, -1)\\;\\big|\\;\\text{treated}\\big],
\\qquad k < -1,$$

which is exactly zero under conditional parallel trends. The TWFE event-study
placebo is run on the same replications, for free, as the standard-practice
comparator.

> **Colab:** upload just this notebook and *Run all*.

In [ ]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

In [ ]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.pretrend_runner import run_named
from did_bcf_revision.metrics import compute_metrics

In [ ]:
REPS = 200      # detection rates need replications; 200 gives MCSE <= 0.035
JOBS = 2        # ~1 GB RAM per worker; drop to 1 if the VM is memory-starved

# WITH_ATT also fits the constrained estimation model on the same replications,
# so the ATT bias the violation causes is measured alongside its detection.
# It doubles the MCMC cost -- worth it at degree 1, skip it at 2 and 3.
WITH_ATT = False

# A Colab session is capped at roughly 8 hours. One fit is ~2 min, so a full
# 200-replication WITH_ATT run is ~7 h at JOBS=2 -- close enough to the cap that
# it is worth splitting. Set these to (0, 100) here and (100, 200) in a second
# copy of this notebook; replications are seeded by index, so the two parts
# concatenate into exactly the undivided run and land in separate files.
REP_START, REP_END = 0, REPS

summaries = run_named(
    "PT_violation_g05",
    linearity_degree=2,
    reps=REPS,
    jobs=JOBS,
    with_att=WITH_ATT,
    rfx="unit",     # unit intercepts absorb the level gap conditional PT allows
    bcf_params=dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3),
    rep_start=REP_START, rep_end=REP_END,
)
summaries.head()

In [ ]:
# `reject05` on the PRE rows is the diagnostic's **size** when the true
# differential slope is 0 and its **detection rate** otherwise; `any_bonf` is the
# per-replication decision rule (any pre-period significant, Bonferroni-scaled).
metrics = compute_metrics(summaries)
pre = metrics[metrics.estimand_type == "PRE"]
pre[["method", "estimand_id", "mean_true", "bias", "cover95",
     "reject05", "mcse_reject05", "role"]].sort_values(["estimand_id", "method"])

## Conditional check

`Delta(k)` within covariate subgroups. A marginal event study cannot produce
this without pre-specifying the interactions; here it comes out of the same fit,
and it is the version that matches what the estimator actually assumes.

In [ ]:
sub = metrics[metrics.estimand_type == "PRE_SUB"]
sub[["estimand_id", "mean_true", "bias", "emp_sd", "cover95", "reject05"]]